# 03 — Modelos Baseline y Evaluación
**CRISP-DM: Modelado + Evaluación** · Etapa 1: *Baseline (≥3 algoritmos) + tabla comparativa + interpretabilidad*

Requiere `pip install -r requirements.txt` (scikit-learn, xgboost, shap).
Usa `data/processed/dataset_modelado.csv` generado en el notebook 02.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))
import pandas as pd
from prediccion_precios import features as ft, evaluation as ev, config
from prediccion_precios import models_baseline as mb, interpretability as it
pd.set_option("display.width",160); pd.set_option("display.max_columns",60)

In [2]:
data = pd.read_csv(config.DATASET_MODELADO, parse_dates=[config.COL_FECHA])
n_antes = len(data)
# dropna=True: descarta el NaN estructural de lags/medias móviles en las
# primeras semanas de cada producto (documentado y tratado en el notebook 02)
X, y = ft.construir_matriz_modelado(data)
X = X.astype(float)
n_descartadas = n_antes - len(X)
print(f"Filas descartadas por NaN estructural (lags/medias móviles iniciales): "
      f"{n_descartadas} ({n_descartadas / n_antes * 100:.2f}%)")
X_train, X_test, y_train, y_test = ev.split_temporal(X, y)
print("X", X.shape, "| train", len(X_train), "| test", len(X_test))

Filas descartadas por NaN estructural (lags/medias móviles iniciales): 40 (4.85%)
X (785, 34) | train 628 | test 157


## 1. Entrenar los 3 baselines (Regresión Lineal, Random Forest, XGBoost)

In [3]:
modelos = mb.entrenar_todos(X_train, y_train)
list(modelos.keys())

['regresion_lineal', 'random_forest', 'xgboost']

## 2. Tabla comparativa de métricas (MAE / RMSE / MAPE / R²)

In [4]:
resultados = {n: ev.calcular_metricas(y_test, m.predict(X_test)) for n, m in modelos.items()}
tabla = ev.tabla_comparativa(resultados)
print("Meta objetivo: MAPE <", config.META_MAPE_OBJETIVO*100, "%")
tabla

Meta objetivo: MAPE < 15.0 %


,MAE,RMSE,MAPE_%,R2
Modelo,,,,
regresion_lineal,1.3334,2.1953,5.1593,0.9946
random_forest,1.7463,2.7345,6.2119,0.9916
xgboost,1.6220,2.5206,6.5363,0.9929


## 3. Validación cruzada temporal (ventana expansiva)
Se pasa la *fábrica* del modelo (`mb.FABRICAS[nombre]`) para re-crearlo en cada fold.

In [5]:
for nombre, fabrica in mb.FABRICAS.items():
    cv = ev.validacion_cruzada_temporal(fabrica, X, y, n_splits=config.CV_SPLITS)
    print(nombre, "-> MAPE %:", cv["MAPE_%"], "| R2:", cv["R2"])

regresion_lineal -> MAPE %: {'media': 7.6634, 'std': 4.8862} | R2: {'media': 0.9709, 'std': 0.0481}
random_forest -> MAPE %: {'media': 8.2272, 'std': 3.6995} | R2: {'media': 0.9546, 'std': 0.0565}
xgboost -> MAPE %: {'media': 9.9944, 'std': 5.0474} | R2: {'media': 0.9056, 'std': 0.1337}


## 4. Interpretabilidad — importancia de variables + SHAP

In [6]:
mejor = modelos["xgboost"]
display(it.importancia_variables(mejor, X.columns).head(15))
it.graficar_importancia(mejor, X.columns)            # -> reports/figures/importancia_variables.png
it.explicar_shap(mejor, X_test)                       # -> reports/figures/shap_summary.png

,feature,importancia
0,precio_lag1,0.327805
1,media_movil_4,0.205008
2,rendimiento_kg_ha,0.169891
3,media_movil_8,0.161939
4,precio_lag8,0.071065
5,precio_lag4,0.030439
6,precio_lag2,0.014549
7,area_ha,0.005622
8,media_movil_12,0.004804
9,produccion_t,0.001747


Background dataset has 157 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=157 when initializing the masker.
PermutationExplainer explainer: 158it [00:38,  4.05it/s]                         
c:\Users\irmal\Downloads\prediccion_precios_agricolas_V2\prediccion\src\prediccion_precios\interpretability.py:77: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_muestra, show=False)


WindowsPath('C:/Users/irmal/Downloads/prediccion_precios_agricolas_V2/prediccion/reports/figures/shap_summary.png')

## 5. Guardar los modelos entrenados

In [7]:
for nombre, modelo in modelos.items():
    mb.guardar_modelo(modelo, nombre)
print("Modelos guardados en", config.MODELS_DIR)

Modelos guardados en C:\Users\irmal\Downloads\prediccion_precios_agricolas_V2\prediccion\models


### Conclusiones de Etapa 1
> Indicar el mejor baseline, su MAPE frente a la meta (<15%), las variables más
> influyentes según SHAP y los próximos pasos hacia la Etapa 2 (optimización de
> hiperparámetros, LSTM/redes neuronales y ensemble).